# Phase 4 — Opportunity Sizing: Annual Margin Gap from Weather-Responsive Pricing at LGA

> **Finding:** Weather-responsive pricing at LGA could capture **\$156K–\$267K in additional annual margin** (mid estimate: \$254K) during light rain events, based on a statistically significant \$0.073/mile margin uplift in the M1 regression model.

**This notebook:**
1. Pulls light rain hour counts and volume stats from processed data
2. Constructs low/mid/high dollar scenarios per the methodology in `docs/business_case_frame.md`
3. Builds a sensitivity table (±10% volume, ±1 SE on coefficient)
4. Produces `chart4_opportunity_estimate.png`
5. Updates `docs/business_case_frame.md` with the dollar figure

**Data source:** `data/processed/fhvhv_lga_hourly_with_weather_2025.parquet` (Jan 1 – Aug 27, 2025)

**Model coefficients:** `outputs/tables/baseline_model_coefficients.csv` (Phase 3 outputs)


In [ ]:
# Standard imports
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Add repo root to path so we can import project modules
REPO_ROOT = Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.models.opportunity_sizing import (
    load_data, assign_weather_regime, get_volume_stats,
    calculate_scenarios, build_sensitivity_table,
    generate_chart, print_interpretation,
    LIGHT_RAIN_COEF_POINT, LIGHT_RAIN_COEF_SE,
    LIGHT_RAIN_COEF_CI_LOWER, LIGHT_RAIN_COEF_CI_UPPER,
    ANN_LOW, ANN_MID, ANN_HIGH,
    OBS_START, OBS_END,
)

print('Imports OK. Running from:', REPO_ROOT)

## Step 1: Coefficient Inputs from Phase 3

We use the **all-season M1 regression** coefficient for light rain — not the original cold-only estimate.

The business case frame (Phase 2) anticipated $0.0847/mile (cold-only M1). The all-season rerun in Phase 3 produced **$0.073/mile**, a ~14% reduction. Phase 4 uses the all-season coefficient per the business case frame's update rule: *"If the new coefficient differs materially, Phase 4 updates all scenarios accordingly."*

In [ ]:
coef_summary = pd.DataFrame({
    'Parameter': [
        'Point estimate ($/mile)',
        'Robust standard error',
        '95% CI lower ($/mile)',
        '95% CI upper ($/mile)',
        'p-value',
        'Model',
        'N observations',
        'Significance',
    ],
    'Value': [
        f'${LIGHT_RAIN_COEF_POINT:.4f}',
        f'${LIGHT_RAIN_COEF_SE:.4f}',
        f'${LIGHT_RAIN_COEF_CI_LOWER:.4f}',
        f'${LIGHT_RAIN_COEF_CI_UPPER:.4f}',
        '< 0.001',
        'M1 (all seasons, HC3 robust SEs)',
        '5,662',
        '*** (p < 0.001)',
    ]
})
print('Light rain coefficient on margin per mile (M1, all-season):')
print(coef_summary.to_string(index=False))

## Step 2: Light Rain Hour Counts and Seasonal Coverage

The dataset covers Jan 1 – Aug 27, 2025 (238 days). The observed 357 light rain hours understate the full-year total because Sep–Dec is typically **rainier** in NYC than Jan–Aug.

**Annualization basis:**
- Calendar coverage ratio: 365 ÷ 238 = 1.533
- Low (1.40×): conservative — assumes fall rain rate ≈ observed Jan–Aug average
- Mid (1.55×): day-ratio with modest fall rain premium
- High (1.70×): optimistic — Sep–Nov estimated ~30% rainier than Jan–Aug avg

Source for seasonal adjustment: NOAA LGA historical monthly precipitation records (NYC avg Sep–Nov is ~20–30% above summer months). See Assumption B4 in `docs/business_case_frame.md`.

In [ ]:
df = load_data()
df = assign_weather_regime(df)

regime_counts = df['weather_regime'].value_counts().rename_axis('Regime').reset_index(name='Hours')
regime_counts['% of dataset'] = (regime_counts['Hours'] / len(df) * 100).round(1)
print(f'Dataset period: {OBS_START} to {OBS_END} ({df.shape[0]} hourly observations)')
print()
print(regime_counts.to_string(index=False))

print()
ann_table = pd.DataFrame({
    'Scenario': ['Low', 'Mid', 'High'],
    'Annualization factor': [ANN_LOW, ANN_MID, ANN_HIGH],
    'Estimated annual light rain hours': [
        round(357 * ANN_LOW, 1),
        round(357 * ANN_MID, 1),
        round(357 * ANN_HIGH, 1),
    ],
    'Basis': [
        'Conservative (Sep–Dec ≈ observed avg)',
        'Day-ratio 365/238 + modest fall premium',
        'Optimistic: Sep–Nov ~30% rainier',
    ]
})
print(ann_table.to_string(index=False))

## Step 3: Trip Volume and Miles by Weather Regime

**Key observation:** Light rain hours have *fewer* trips than no-rain hours (529 vs 556 per hour). Rain at LGA slightly suppresses trip volume — likely due to driver supply constraints. The High scenario is not elevated via a volume premium; it is elevated via the 1.70× annualization factor.

This is an important note: if rain *reduced* trip volume by more than assumed, the opportunity would shrink. Conversely, if weather-responsive pricing reduces trip volume further (demand elasticity), the opportunity could be lower than modeled.

In [ ]:
stats = get_volume_stats(df)

vol_table = pd.DataFrame({
    'Regime': ['No rain (baseline)', 'Light rain', 'Heavy rain'],
    'Hours in dataset': [
        len(df[df['weather_regime'] == 'no_rain']),
        len(df[df['weather_regime'] == 'light_rain']),
        len(df[df['weather_regime'] == 'heavy_rain']),
    ],
    'Avg trips/hr': [
        round(df[df['weather_regime']=='no_rain']['request_count'].mean(), 1),
        round(df[df['weather_regime']=='light_rain']['request_count'].mean(), 1),
        round(df[df['weather_regime']=='heavy_rain']['request_count'].mean(), 1),
    ],
    'Avg trip miles': [
        round(df[df['weather_regime']=='no_rain']['trip_miles_mean'].mean(), 3),
        round(df[df['weather_regime']=='light_rain']['trip_miles_mean'].mean(), 3),
        round(df[df['weather_regime']=='heavy_rain']['trip_miles_mean'].mean(), 3),
    ],
})
print(vol_table.to_string(index=False))
print()
print(f'*** Rain-period volume ({stats["rain_trips_per_hour"]:.1f}/hr) is '
      f'{stats["baseline_trips_per_hour"] - stats["rain_trips_per_hour"]:.1f} trips/hr '
      f'BELOW no-rain baseline ({stats["baseline_trips_per_hour"]:.1f}/hr).')
print('*** Assumption B6 confirmed: trip miles are nearly identical across regimes (~11.3 miles).')

## Step 4: Annual Opportunity — Low / Mid / High Scenarios

**Formula:**
```
Annual Opportunity ($) =
    (obs_light_rain_hours × annualization_factor)
    × avg_trips_per_rain_hour
    × avg_trip_miles
    × margin_uplift_per_mile
```

In [ ]:
scenarios_df = calculate_scenarios(stats)

display_cols = [
    'Scenario',
    'Estimated annual light rain hours',
    'Avg trips per rain hour',
    'Avg trip miles',
    'Margin uplift coefficient ($/mile)',
    'Annual margin opportunity (USD)',
]
print(scenarios_df[display_cols].to_string(index=False))
print()

low_val = scenarios_df.loc[scenarios_df['Scenario']=='Low', 'Annual margin opportunity (USD)'].iloc[0]
mid_val = scenarios_df.loc[scenarios_df['Scenario']=='Mid', 'Annual margin opportunity (USD)'].iloc[0]
high_val = scenarios_df.loc[scenarios_df['Scenario']=='High', 'Annual margin opportunity (USD)'].iloc[0]

print(f'>>> DOLLAR FIGURE: ${low_val:,.0f} – ${high_val:,.0f} annually (mid: ${mid_val:,.0f})')

## Step 5: Sensitivity Analysis

Anchored on Mid scenario (1.55× annualization, baseline trips, baseline miles). Shows how the estimate changes under ±10% trip volume and ±1 standard error shift on the coefficient.

In [ ]:
sensitivity_df = build_sensitivity_table(stats)
print(sensitivity_df.to_string(index=False))

min_opp = sensitivity_df['Annual opportunity (USD)'].min()
max_opp = sensitivity_df['Annual opportunity (USD)'].max()
print(f'\nSensitivity range: ${min_opp:,.0f} – ${max_opp:,.0f}')
print('Interpretation: even under pessimistic conditions (-10% volume, -1 SE coefficient),'
      f' annual opportunity remains above ${min_opp:,.0f}.')

## Step 6: Summary Chart

In [ ]:
chart_path = Path('outputs/charts/chart4_opportunity_estimate.png')
generate_chart(scenarios_df, chart_path)
print(f'Chart saved: {chart_path}')

# Display inline if running interactively
from IPython.display import Image, display
display(Image(str(chart_path)))

## Step 7: Plain-English Interpretation

In [ ]:
interp = print_interpretation(scenarios_df)
print(interp)

## Step 8: Save All Outputs

Outputs are saved to:
- `outputs/tables/opportunity_sizing_scenarios.csv`
- `outputs/tables/sensitivity_analysis.csv`
- `outputs/charts/chart4_opportunity_estimate.png`
- `docs/business_case_frame.md` (updated with dollar figures)

In [ ]:
from pathlib import Path

OUT_TABLES = Path('outputs/tables')
scenarios_df.to_csv(OUT_TABLES / 'opportunity_sizing_scenarios.csv', index=False)
sensitivity_df.to_csv(OUT_TABLES / 'sensitivity_analysis.csv', index=False)
print('CSVs saved.')

print('\nAll Phase 4 outputs confirmed:')
for p in [
    'outputs/tables/opportunity_sizing_scenarios.csv',
    'outputs/tables/sensitivity_analysis.csv',
    'outputs/charts/chart4_opportunity_estimate.png',
]:
    exists = Path(p).exists()
    print(f'  [{"OK" if exists else "MISSING"}] {p}')